# Day 5: Building With LangChain

So far we have built loops, context, plans, and memory mostly by hand. Today we see how **LangChain** gives us these pieces ready-made.

**Today's plan:**
1. **LCEL**: snap a prompt, a model, and a parser together like building blocks
2. **Tools**: turn any Python function into something the agent can use
3. **One agent step by hand**: see exactly what happens inside the loop
4. **Lab**: build an agent with a search tool, a calculator, and a custom tool, and check that it picks the right tool each time

### Before you start

Create a file called `.env` in the same folder as this notebook and add your Azure OpenAI details:

```
AZURE_OPENAI_ENDPOINT=https://<your-resource>.openai.azure.com/
AZURE_OPENAI_API_KEY=<your-key>
AZURE_OPENAI_DEPLOYMENT=<your-chat-deployment-name>
AZURE_OPENAI_API_VERSION=2024-10-21
```

Never share this file or upload it to GitHub, because it contains your secret key.

### Setup
This cell loads your `.env` values and creates the `llm` object we use in the rest of this notebook. Run it first.

In [ ]:
# pip install -U langchain langgraph langchain-openai python-dotenv ddgs

import os
from typing import TypedDict
from dotenv import load_dotenv

from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langgraph.graph import StateGraph, START, END

# 1. Load the values from the .env file
load_dotenv()

# 2. Make sure nothing important is missing
required = ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_DEPLOYMENT"]
missing = [name for name in required if not os.environ.get(name)]
if missing:
    raise ValueError(f"Please add these to your .env file: {missing}")

# 3. Create the LLM
llm = AzureChatOpenAI(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT"],
    api_version=os.environ.get("AZURE_OPENAI_API_VERSION", "2024-10-21"),
    temperature=0,
)

print("Setup done. Using deployment:", os.environ["AZURE_OPENAI_DEPLOYMENT"])

In [ ]:
# Quick check: can we talk to the model?
print(llm.invoke("Say hello to the class in one short sentence.").content)

## 1. LCEL: the building blocks

LCEL uses the `|` symbol to connect pieces, like a pipe:

```
prompt  |  llm  |  parser
```

Every piece works the same way, so any chain can:

- `invoke`: run once
- `batch`: run on many inputs at the same time
- `stream`: show the answer word by word

In [ ]:
prompt = ChatPromptTemplate.from_template("Explain {topic} to a 10-year-old in 2 sentences.")
chain = prompt | llm | StrOutputParser()

print(chain.invoke({"topic": "black holes"}))

In [ ]:
# batch: several inputs at once
for answer in chain.batch([{"topic": "rainbows"}, {"topic": "volcanoes"}]):
    print("-", answer)

In [ ]:
# stream: prints as the model writes
for piece in chain.stream({"topic": "gravity"}):
    print(piece, end="", flush=True)

## 2. Making a tool with `@tool`

Add `@tool` above a normal Python function, and the function becomes a tool.

**Important:** the agent never reads your code. It only sees the **name**, the **description** (your docstring), and the **inputs**. A vague description is the number one reason an agent picks the wrong tool.

In [ ]:
from langchain.tools import tool

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers."""
    return a * b

print("Name       :", multiply.name)
print("Description:", multiply.description)
print("Inputs     :", multiply.args)
print("Result     :", multiply.invoke({"a": 12.5, "b": 8}))

## 3. One agent step, by hand

This is the same loop from Day 1, broken into four visible steps:

| Step | What happens |
|---|---|
| 1. Attach tools | `bind_tools` tells the model which tools exist |
| 2. Model decides | The model replies with a **tool call** instead of a normal answer |
| 3. Tool runs | **We** run the tool (the model never runs code itself) |
| 4. Result comes back | We send the result to the model, and it writes the final answer |

In [ ]:
llm_with_tools = llm.bind_tools([multiply])                # 1. attach tools

messages = [("user", "What is 12.5 times 8?")]
ai_msg = llm_with_tools.invoke(messages)                   # 2. model decides
print("Model wants to call:", ai_msg.tool_calls)

tool_msg = multiply.invoke(ai_msg.tool_calls[0])           # 3. tool runs
print("Tool returned:", tool_msg.content)

final = llm_with_tools.invoke(messages + [ai_msg, tool_msg])   # 4. result goes back
print("Final answer:", final.content)

## 4. A tool with several inputs: `StructuredTool`

When a tool needs several clearly typed inputs, we describe each input with a Pydantic class. Every field gets its own description, so the model knows exactly what to pass.

Our custom tool calculates a monthly loan **EMI**.

In [ ]:
from pydantic import BaseModel, Field
from langchain_core.tools import StructuredTool

class EMIInput(BaseModel):
    principal: float = Field(description="Loan amount in rupees")
    annual_rate: float = Field(description="Yearly interest rate in percent, e.g. 9")
    months: int = Field(description="Loan duration in months")

def calculate_emi(principal, annual_rate, months):
    r = annual_rate / 12 / 100
    return round(principal * r * (1 + r) ** months / ((1 + r) ** months - 1), 2)

emi_tool = StructuredTool.from_function(
    func=calculate_emi,
    name="emi_calculator",
    description="Calculate monthly loan EMI from principal, yearly rate and months.",
    args_schema=EMIInput,
)
print("EMI:", emi_tool.invoke({"principal": 500000, "annual_rate": 9, "months": 60}))

## 5. Lab tools: search, calculator, and our EMI tool

Look at how each description says clearly **when** to use the tool. That's what helps the agent tell them apart.

LangChain also has **toolkits**, ready-made bundles of related tools (for example, for SQL databases or files). We build our own tools here so every line is visible.

In [ ]:
from ddgs import DDGS   # free DuckDuckGo search, no API key needed

@tool
def web_search(query: str) -> str:
    """Search the internet for CURRENT facts or news. Input: a short search query."""
    results = DDGS().text(query, max_results=3)
    return "\n".join(r["title"] + ": " + r["body"] for r in results)

@tool
def calculator(expression: str) -> str:
    """Solve a math expression like '25 * 4 + 10'. Use for ANY arithmetic."""
    return str(eval(expression, {"__builtins__": {}}))   # OK for class demos, not production

tools = [web_search, calculator, emi_tool]
print("Tools:", [t.name for t in tools])

## 6. Build the agent

`create_agent` runs the four steps from section 3 **in a loop** for us, until the model stops asking for tools. (It replaces the older `AgentExecutor`, and it's built on LangGraph.)

`show_steps` prints every tool call, so we can see what the agent decided.

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    llm,
    tools=tools,
    system_prompt="You are a helpful assistant. Always use a tool for math, loans and current facts.",
)

def show_steps(result):
    """Print which tools the agent used, step by step."""
    for msg in result["messages"]:
        if msg.type == "ai" and msg.tool_calls:
            for call in msg.tool_calls:
                print("  TOOL CALL ->", call["name"], call["args"])
        elif msg.type == "tool":
            print("  RESULT    <-", msg.content[:80])
    print("  ANSWER:", result["messages"][-1].content)

## 7. Hands-on lab: does it pick the right tool?

The first three questions each need **one** tool. The last question needs **two tools, one after the other**: search for the rate first, then calculate the EMI.

For each question we list the tools we expect, and the code marks it PASS or FAIL.

In [ ]:
tests = [
    ("What is 1234 * 56 / 7?",                        ["calculator"]),
    ("Who won the most recent IPL final?",            ["web_search"]),
    ("EMI for a 5 lakh loan at 9% for 60 months?",    ["emi_calculator"]),
    ("Find the current RBI repo rate, then calculate the EMI for "
     "10 lakh over 60 months at that rate plus 3%.",   ["web_search", "emi_calculator"]),
]

for question, expected_tools in tests:
    print("\nQ:", question)
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    show_steps(result)

    used = [c["name"] for m in result["messages"] if m.type == "ai" for c in m.tool_calls]
    ok = all(t in used for t in expected_tools)
    print("  PASS" if ok else f"  FAIL: expected {expected_tools}, used {used}")

## Key takeaways

- **LCEL** connects blocks with `|`, and every block supports `invoke`, `batch`, and `stream`.
- **The description is what the agent sees.** Write it clearly and make each tool easy to tell apart.
- **An agent is just a loop:** attach tools, the model decides, the tool runs, and the result comes back. `create_agent` runs that loop for you.
- **Always look at the steps** to confirm the agent used the right tools, not just that the final answer looks fine.

**Try this:** Change the calculator's docstring to just `"A tool."` and run the lab again. Does the agent still choose it?